# Symbolic Regression Hackathon Prompt

2026 IAIFI Summer School

# Equation Hunting in Gaia

## The physics

Gaia gives the information about where millions of stars are and how fast they are moving, which is a
sample from a phase-space density $f(\mathbf{x}, \mathbf{v})$. 
An open question is what gravitational potential $\Phi(\mathbf{x})$ is bending all of those orbits, which has no closed form available. If we suppose the Galaxy is close to a steady state then $f$ and $\Phi$ are related by the collisionless Boltzmann equation,

$$\mathbf{v}\cdot\nabla_{\mathbf x} f \;-\; \nabla_{\mathbf x}\Phi\cdot\nabla_{\mathbf v} f \;=\; 0,$$

which just says that the density around a star moving through phase space does not change along
its own orbit. Stellar accelerations here are of order 1 cm/s/yr, far
below anything we could ever measure directly on one star, and yet a whole catalogue of stars
sampling one density is enough to constrain the field that is pulling on all of them at once.

## The task
The goal is to find a pair of functions $f$ and $\Phi$ that satisfy the collisionless Boltzmann equation and accurately describe the observed stellar distribution. This is a challenging inverse problem, as we are trying to infer the underlying potential from the observed density.

Using the template-and-derivative machinery in SR, plus whatever preprocessing you find
useful, search jointly for a closed-form $f$ and $\Phi$ that drive that residual to zero while $f$
still describes the stars you were actually given. Existing work models $\Phi$ with a neural network, so a compact symbolic form that competes with that would be new.

This notebook gets you to the starting line with a local, clean sample of stars with the columns
you need for approaching the problem.

In [ ]:
import os
os.environ.setdefault("PYTHON_JULIACALL_THREADS", "auto")
os.environ.setdefault("PYTHON_JULIACALL_AUTOLOAD_IPYTHON_EXTENSION", "no")

import gc
import shutil
import urllib.request
from pathlib import Path

import numpy as np
import pandas as pd

import hackathon_data as hk
from hackathon_data import plot_cut, plot_phase_space

# For reproducibility, we can set the following parameters:
REPRO = dict(deterministic=True, parallelism="serial", verbosity=0, progress=False)
# You can turn them off, which generally makes things faster.

## 1. Getting the data

The catalogue lives on Zenodo as a single pickled `pandas` DataFrame, about 33 million stars with
full 6D phase-space information, propagated uncertainties and quality flags, in three coordinate
frames at once (equatorial, galactic, galactocentric). It is processed from the raw Gaia DR3
archive by Cian Roche's pipeline:

- record: https://doi.org/10.5281/zenodo.8088365
- pipeline and full column documentation: https://github.com/CianMRoche/GAIA-DR3-6D-Kinematics

The file itself is 10.5 GB, and loading the whole thing into `pandas` will want on the order of
that much RAM again while it unpickles, so the cell below downloads it once to a local cache and
every cell after that works from a neighbourhood cut small enough to live on a laptop. If 10.5 GB
is not something your connection or your disk wants to take on right now, skip straight to
section 2 and swap in whatever local subsample your table asks you to start from instead, the rest
of the notebook only assumes you end up with the columns listed there.

In [ ]:
DATA = Path("data")
DATA.mkdir(exist_ok=True)
PKL = DATA / "DR3_6D_kinematics.pkl.part"
URL = "https://zenodo.org/records/8088365/files/DR3_6D_kinematics.pkl?download=1"


def download(url: str, dest: Path, chunk: int = 1 << 20) -> None:
    if dest.exists():
        print(f"{dest} already on disk ({dest.stat().st_size / 1e9:.1f} GB), skipping download")
        return
    tmp = dest.with_suffix(dest.suffix + ".part")
    with urllib.request.urlopen(url, timeout=60) as resp, open(tmp, "wb") as f:
        total = int(resp.headers.get("Content-Length", 0))
        done = 0
        while block := resp.read(chunk):
            f.write(block)
            done += len(block)
            if total:
                print(f"\r{done / 1e9:6.2f} / {total / 1e9:.2f} GB", end="", flush=True)
    shutil.move(tmp, dest)
    print()


download(URL, PKL)

In [ ]:
raw = pd.read_pickle(PKL)
print(f"{len(raw):,} stars, {raw.memory_usage(deep=True).sum() / 1e9:.1f} GB in memory")
raw.columns.tolist()

## 2. What is in each row

The columns come in four groups, each repeated across the three frames the pipeline carries.

| group | columns | notes |
|---|---|---|
| identity | `source_id`, `feH` | `feH` is metallicity, not used below |
| position | `ra, dec` (deg), `l, b` (deg), `x, y, z` (kpc, galactocentric), `r, r_helio` (kpc), `parallax`, `parallax_nozpcorr` | `r_helio` is plain distance from us, and does not depend on any convention for where the Galactic centre sits |
| velocity | `pmra, pmdec` (mas/yr), `vx, vy, vz` (km/s, galactocentric), `vr, vtheta, vphi` (km/s, galactocentric spherical), `v_radial, vabs` (km/s) | `vx, vy, vz` is the frame the collisionless Boltzmann equation above is written in |
| uncertainty & quality | `*_err` / `*_error` for every position and velocity column above, plus `plx_over_error`, `ruwe`, `rv_nb_transits`, `rv_expected_sig_to_noise` | propagated uncertainties are there for you to use as fit weights, the way session 4 weighted by density |

Everything the collisionless Boltzmann equation needs is `x, y, z, vx, vy, vz` and their
uncertainties, all already in one consistent Cartesian galactocentric frame, which is why those
are the columns the cut below keeps.

## 3. Cutting to a neighbourhood you can actually search

Two things have to both be true of whatever region you fit on. It has to hold enough stars that
$f(\mathbf{x}, \mathbf{v})$ is a density you can actually estimate rather than a handful of
points, and it has to be small enough that $\Phi(\mathbf{x})$ stands a chance of being something a
symbolic search can find.

For example, the cut below is on `r_helio`, plain heliocentric distance,  because it does not
require trusting any convention for where the Sun sits in the galactocentric frame, unlike a cut
written directly in `x, y, z`. A basic quality filter allows to use `ruwe` flags astrometry that
does not fit a single-star model well, and `rv_nb_transits` flags radial velocities built from too
few visits to trust. Positions and velocities are then recentred on the sample's own mean, which
only shifts where $\Phi$'s expansion point sits and changes none of its gradients, so the residual
above is unaffected. Treat the two numbers below as knobs, not constants.

In [ ]:
R_CUT = 0.5     # kpc, heliocentric radius kept
RUWE_MAX = 1.4   # astrometric quality, lower is better

quality = (
    (raw["r_helio"] < R_CUT)
    & (raw["ruwe"] < RUWE_MAX)
    & raw["vx"].notna() & raw["vy"].notna() & raw["vz"].notna()
)

cols = ["x", "y", "z", "vx", "vy", "vz",
        "x_err", "y_err", "z_err", "vx_err", "vy_err", "vz_err"]
local = raw.loc[quality, cols].reset_index(drop=True)

fig = plot_cut(raw.loc[quality, "x"], raw.loc[quality, "y"], raw.loc[quality, "z"],
               raw.loc[quality, "r_helio"], R_CUT,
               title=f"{len(local):,} stars kept out of {len(raw):,}")

del raw
gc.collect()

In [ ]:
centre = local[["x", "y", "z"]].mean()
for c in ["x", "y", "z"]:
    local[c] = local[c] - centre[c]

CACHE = DATA / "local_sample.parquet"
local.to_parquet(CACHE)
print(f"cached {len(local):,} rows to {CACHE}")
local.describe()

If you come back to this later, `local = pd.read_parquet("data/local_sample.parquet")` gets you
straight back here without touching the 10.5 GB file again.

In [ ]:
fig1, fig2 = plot_phase_space(local.x, local.y, local.z, local.vx, local.vy, local.vz)

## 4. Holding out a test split

The same plain random split every session has used, because the point of it is the same here as
everywhere else: whatever residual you drive to zero on the fit rows, check it on rows the search
never saw before you believe a symbolic $f$ and $\Phi$ over the network everyone else is using.

In [ ]:
def split(df: pd.DataFrame, test_size: float = 0.2, seed: int = 0):
    rng = np.random.default_rng(seed)
    order = rng.permutation(len(df))
    cut = int(round(len(df) * (1 - test_size)))
    return np.sort(order[:cut]), np.sort(order[cut:])


fit_idx, test_idx = split(local)
fit, test = local.iloc[fit_idx].reset_index(drop=True), local.iloc[test_idx].reset_index(drop=True)
print(f"{len(fit):,} to fit on, {len(test):,} held out")

## Where this leaves you

`fit` and `test` are DataFrames with `x, y, z, vx, vy, vz` and their `_err` columns, recentred on
the sample and drawn from a shell close enough to the Sun that a compact $\Phi$ has a chance of
being a good description of it.